In [32]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from typing import TypedDict

In [33]:
class BatsMenState(TypedDict):
    runs:int
    balls:int
    fours:int
    sixes:int
    strikeRate:float
    bpb:float
    boundaryPercentage:float
    summary:str

In [34]:
#Strikerate Node
def ballsPerBoundary(state:BatsMenState)->BatsMenState:
    
    fours=state['fours']
    sixes=state['sixes']
    balls=state['balls']
    bpb=(balls/(fours+sixes))
    return {
        "bpb": bpb
    }

In [35]:
#Strikerate Node
def StrikeRate(state:BatsMenState)->BatsMenState:
    runs=state['runs']
    balls=state['balls']
    strike_rate=(runs/balls)*100
    return {
        "strikeRate": strike_rate
    }

In [36]:
#Strikerate Node
def boundaryPercentage(state:BatsMenState)->BatsMenState:
    fours=state['fours']
    sixes=state['sixes']
    runs=state['runs']
    boundary_percentage = (
        ((fours * 4 + sixes * 6) / runs) * 100
        if runs > 0
        else 0
    )

    return {
        "boundaryPercentage": boundary_percentage
    }

In [37]:
def summary_node(state: BatsMenState) -> dict:
    summary = (
        f"The batter scored {state['runs']} runs off {state['balls']} balls "
        f"at a strike rate of {state['strikeRate']:.2f}. "
        f"He hit {state['fours']} fours and {state['sixes']} sixes. "
        f"His boundary percentage was {state['boundaryPercentage']:.2f}%, "
        f"with {state['bpb']:.2f} balls per boundary."
    )
    

    return {
        "summary": summary
    }

In [38]:
#define graph
graph=StateGraph(BatsMenState)

#add nodes
graph.add_node("strikeRate",StrikeRate)
graph.add_node("ballsPerBoundary",ballsPerBoundary)
graph.add_node("boundaryPercentage",boundaryPercentage)
graph.add_node("summary_node",summary_node)

#add edges
graph.add_edge(START,"strikeRate")
graph.add_edge(START,"ballsPerBoundary")
graph.add_edge(START,"boundaryPercentage")
graph.add_edge("strikeRate","summary_node")
graph.add_edge("ballsPerBoundary","summary_node")
graph.add_edge("boundaryPercentage","summary_node")
graph.add_edge("summary_node",END)

#compile
workflow=graph.compile()

In [39]:
initial_state={"runs":20,"balls":10,"fours":3,"sixes":1}
final_state=workflow.invoke(initial_state)
print(final_state)

{'runs': 20, 'balls': 10, 'fours': 3, 'sixes': 1, 'strikeRate': 200.0, 'bpb': 2.5, 'boundaryPercentage': 90.0, 'summary': 'The batter scored 20 runs off 10 balls at a strike rate of 200.00. He hit 3 fours and 1 sixes. His boundary percentage was 90.00%, with 2.50 balls per boundary.'}
